# Análisis de factores desencadenantes de ideación suicida

Este Notebook analiza los factores desencadenantes registrados en casos de ideación suicida notificados en adolescentes de 12 a 17 años en Bogotá entre 2017 y 2025.

En esta primera sección se carga la base original, se valida su estructura y se aplican directamente los filtros del estudio. Posteriormente, estas operaciones serán ejecutadas mediante las funciones definidas en los módulos `.py`.

In [ ]:
# Librerías para manipulación de datos y manejo de rutas
from pathlib import Path
import pandas as pd

# Mostrar todas las columnas al visualizar un DataFrame
pd.set_option("display.max_columns", None)

In [ ]:
# Nombre del archivo que contiene la base de datos
ruta_archivo = Path("osb_salud_mental_ideacion_e_intento.csv")

# Verificar que el archivo exista antes de intentar cargarlo
if not ruta_archivo.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo: {ruta_archivo.resolve()}"
    )

print("Archivo encontrado correctamente.")
print("Ruta:", ruta_archivo.resolve())

In [ ]:
# Cargar el CSV usando punto y coma como separador
data_original = pd.read_csv(
    ruta_archivo,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

print("Base cargada correctamente.")
print(f"Número de registros: {data_original.shape[0]:,}")
print(f"Número de columnas: {data_original.shape[1]}")

data_original.head()

In [ ]:
# Visualizar los nombres originales de las columnas
print("Columnas disponibles en la base:\n")

for numero, columna in enumerate(data_original.columns, start=1):
    print(f"{numero}. {columna}")

In [ ]:
# Columnas necesarias para aplicar los filtros del estudio
columnas_filtro = [
    "ano_notificacion",
    "ciclovital",
    "clasificaciondelaconducta"
]

# Nueve factores desencadenantes seleccionados
factores = [
    "enfermedades_dolorosas",
    "maltrato_sexual",
    "muerte_familiar",
    "conflicto_pareja",
    "problemas_economicos",
    "esc_educ",
    "problemas_juridicos",
    "problemas_laborales",
    "suicidio_amigo"
]

columnas_requeridas = columnas_filtro + factores
columnas_faltantes = [
    columna for columna in columnas_requeridas
    if columna not in data_original.columns
]

if columnas_faltantes:
    raise ValueError(f"Faltan columnas requeridas: {columnas_faltantes}")

print("Validación exitosa.")
print("Las columnas de filtro y los nueve factores están disponibles.")

In [ ]:
# Crear una copia para conservar intacta la base original
data = data_original.copy()

# Eliminar espacios adicionales en las variables de texto utilizadas
data["clasificaciondelaconducta"] = (
    data["clasificaciondelaconducta"].astype("string").str.strip()
)
data["ciclovital"] = (
    data["ciclovital"].astype("string").str.strip()
)

# Convertir el año a formato numérico
data["ano_notificacion"] = pd.to_numeric(
    data["ano_notificacion"], errors="coerce"
)

print("Datos preparados correctamente.")

In [ ]:
# Condiciones definidas en la propuesta del proyecto
condicion_conducta = (
    data["clasificaciondelaconducta"].str.casefold()
    == "Ideación suicida".casefold()
)
condicion_ciclo = (
    data["ciclovital"].str.casefold()
    == "12 – 17 Adolescencia".casefold()
)
condicion_periodo = data["ano_notificacion"].between(
    2017, 2025, inclusive="both"
)

# Aplicar simultáneamente las tres condiciones
data_filtrada = data.loc[
    condicion_conducta & condicion_ciclo & condicion_periodo
].copy()

data_filtrada = data_filtrada.sort_values(
    by="ano_notificacion"
).reset_index(drop=True)

print("Filtros aplicados correctamente.")
print(f"Registros originales: {len(data):,}")
print(f"Registros filtrados: {len(data_filtrada):,}")

In [ ]:
# Contar los registros filtrados correspondientes a cada año
registros_por_ano = (
    data_filtrada["ano_notificacion"]
    .value_counts()
    .sort_index()
    .rename_axis("Año")
    .reset_index(name="Número de registros")
)

registros_por_ano

In [ ]:
# Valores esperados después de aplicar los filtros
total_esperado = 53887
anos_esperados = list(range(2017, 2026))

anos_obtenidos = sorted(
    data_filtrada["ano_notificacion"]
    .dropna().astype(int).unique().tolist()
)

assert len(data_filtrada) == total_esperado, (
    f"Se esperaban {total_esperado:,} registros, "
    f"pero se obtuvieron {len(data_filtrada):,}."
)
assert anos_obtenidos == anos_esperados, (
    f"Los años obtenidos no corresponden a 2017–2025: {anos_obtenidos}"
)
assert data_filtrada["clasificaciondelaconducta"].str.casefold().eq(
    "Ideación suicida".casefold()
).all(), "Existen registros diferentes de ideación suicida."
assert data_filtrada["ciclovital"].str.casefold().eq(
    "12 – 17 Adolescencia".casefold()
).all(), "Existen registros fuera del ciclo vital seleccionado."

print("Todas las verificaciones fueron superadas.")
print(f"La base de análisis contiene {len(data_filtrada):,} registros.")
print("Periodo confirmado: 2017–2025.")
print("Conducta confirmada: Ideación suicida.")
print("Ciclo vital confirmado: 12–17 Adolescencia.")